# Modelo de Recomendação — SVD + Similaridade de Itens

Abordagem: fatoração de matrizes com TruncatedSVD (sklearn) para obter vetores latentes dos filmes, seguida de similaridade de cosseno entre esses vetores para recomendar filmes similares ao histórico do usuário.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
from tqdm import tqdm

PROC_DIR = Path("../data/processed")

# --- Parâmetros ajustáveis ---
N_COMPONENTS        = 50   # fatores latentes do SVD
MIN_USER_RATINGS    = 20   # usuários com menos avaliações são removidos
MIN_MOVIE_RATINGS   = 50   # filmes com menos avaliações são removidos
TOP_N               = 10   # filmes recomendados por usuário
TOP_K_SIMILAR       = 20   # vizinhos mais próximos por filme avaliado
RANDOM_STATE        = 42

## 2. Carga dos dados

In [2]:
ratings = pd.read_csv(PROC_DIR / "ratings_clean.csv", parse_dates=["date"])
movies  = pd.read_csv(PROC_DIR / "movies_clean.csv")

print(f"Ratings carregados : {len(ratings):,}")
print(f"Filmes carregados  : {len(movies):,}")
ratings.head()

Ratings carregados : 25,000,095
Filmes carregados  : 59,047


,userId,movieId,rating,date
0,1,296,5.0,2006-05-17 15:34:04
1,1,306,3.5,2006-05-17 12:26:57
2,1,307,5.0,2006-05-17 12:27:08
3,1,665,5.0,2006-05-17 15:13:40
4,1,899,3.5,2006-05-17 12:21:50


## 3. Filtragem de usuários e filmes com poucos dados

Usuários e filmes com poucas avaliações geram vetores latentes pouco confiáveis.
Removê-los reduz ruído sem perder os padrões principais.

In [3]:
# Filtra usuários
user_counts  = ratings["userId"].value_counts()
valid_users  = user_counts[user_counts >= MIN_USER_RATINGS].index
ratings = ratings[ratings["userId"].isin(valid_users)]

# Filtra filmes
movie_counts = ratings["movieId"].value_counts()
valid_movies = movie_counts[movie_counts >= MIN_MOVIE_RATINGS].index
ratings = ratings[ratings["movieId"].isin(valid_movies)]

print(f"Após filtragem:")
print(f"  Usuários : {ratings['userId'].nunique():,}")
print(f"  Filmes   : {ratings['movieId'].nunique():,}")
print(f"  Ratings  : {len(ratings):,}")

Após filtragem:
  Usuários : 162,540
  Filmes   : 13,176
  Ratings  : 24,644,928


## 4. Divisão treino / teste (split temporal)

Ordenamos por data e usamos os 80% mais antigos para treino e os 20% mais recentes para teste.
Isso simula o cenário real: o modelo só vê avaliações passadas para recomendar filmes futuros.

In [4]:
ratings_sorted = ratings.sort_values("date")
split_idx      = int(len(ratings_sorted) * 0.8)
train          = ratings_sorted.iloc[:split_idx]
test           = ratings_sorted.iloc[split_idx:]

print(f"Treino : {len(train):,} ratings  (até {train['date'].max().date()})")
print(f"Teste  : {len(test):,}  ratings  (a partir de {test['date'].min().date()})")

Treino : 19,715,942 ratings  (até 2016-05-28)
Teste  : 4,928,986  ratings  (a partir de 2016-05-28)


## 5. Matriz esparsa usuário × filme

Reindexamos `userId` e `movieId` para índices contínuos (0, 1, 2, ...) e construímos
uma `csr_matrix` do scipy — formato eficiente para matrizes com muitos zeros.

In [5]:
user_idx  = {int(uid): i for i, uid in enumerate(train["userId"].unique())}
movie_idx = {int(mid): i for i, mid in enumerate(train["movieId"].unique())}
idx_movie = {i: mid for mid, i in movie_idx.items()}

rows = train["userId"].map(user_idx).to_numpy(dtype=int)
cols = train["movieId"].map(movie_idx).to_numpy(dtype=int)
data = train["rating"].to_numpy(dtype=float)

train_matrix = csr_matrix((data, (rows, cols)), shape=(len(user_idx), len(movie_idx)))

n_users, n_movies = len(user_idx), len(movie_idx)
print(f"Matriz: {n_users:,} usuários × {n_movies:,} filmes")
print(f"Esparsidade: {1 - train_matrix.nnz / (n_users * n_movies):.2%}")

# Centralização por usuário: subtrai a média de cada usuário apenas das entradas observadas
user_sums   = np.array(train_matrix.sum(axis=1)).flatten()
user_counts = np.diff(train_matrix.indptr)
user_means  = user_sums / np.maximum(user_counts, 1)

train_matrix_centered = train_matrix.copy().astype(float)
rows_nz = train_matrix_centered.nonzero()[0]          # índice de linha de cada entrada não-zero
train_matrix_centered.data -= user_means[rows_nz]     # subtrai a média do usuário correspondente

print(f"\nMédia global antes da centralização : {data.mean():.3f}")
print(f"Média global após centralização     : {train_matrix_centered.data.mean():.3f}")

Matriz: 137,168 usuários × 12,315 filmes
Esparsidade: 98.83%

Média global antes da centralização : 3.531
Média global após centralização     : -0.000


## 6. SVD — aprendendo os fatores latentes

**O que o SVD faz:** comprime a matriz usuário × filme em duas matrizes menores:
- `user_factors` (usuário × N_COMPONENTS): perfil latente de cada usuário
- `item_factors` (filme × N_COMPONENTS): perfil latente de cada filme

Os N_COMPONENTS fatores representam conceitos como "gosta de animação", "prefere drama", etc.
Mesmo filmes com poucas avaliações ganham um vetor denso por generalização.

In [6]:
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
user_factors = svd.fit_transform(train_matrix_centered)   # matriz centralizada
item_factors = svd.components_.T                           # (n_movies × N_COMPONENTS)

variancia_explicada = svd.explained_variance_ratio_.sum()
print(f"Variância explicada pelos {N_COMPONENTS} componentes: {variancia_explicada:.2%}")
print(f"item_factors shape: {item_factors.shape}")

Variância explicada pelos 50 componentes: 18.10%
item_factors shape: (12315, 50)


## 7. Similaridade de cosseno entre itens

Com os vetores latentes densos, calculamos a similaridade entre todos os pares de filmes.
Resultado: matriz (n_filmes × n_filmes) onde cada célula [i, j] vai de 0 a 1.

In [7]:
item_sim = cosine_similarity(item_factors)  # (n_movies × n_movies)
print(f"Matriz de similaridade: {item_sim.shape}")

# Exemplo: 5 filmes mais similares ao filme de índice 0
ex_idx   = 0
ex_movie = movies[movies["movieId"] == idx_movie[ex_idx]]["title"].values[0]
top5_idx = np.argsort(item_sim[ex_idx])[::-1][1:6]
top5_ids = [idx_movie[i] for i in top5_idx]

print(f"\nFilme base: {ex_movie}")
print("Filmes similares:")
for idx, mid in zip(top5_idx, top5_ids):
    title = movies[movies["movieId"] == mid]["title"].values[0]
    print(f"  {title}  (sim={item_sim[ex_idx][idx]:.3f})")

Matriz de similaridade: (12315, 12315)

Filme base: Fish Called Wanda, A (1988)
Filmes similares:
  Monty Python's And Now for Something Completely Different (1971)  (sim=0.796)
  Monty Python's Life of Brian (1979)  (sim=0.764)
  Monty Python Live at the Hollywood Bowl (1982)  (sim=0.735)
  Monty Python and the Holy Grail (1975)  (sim=0.710)
  Commitments, The (1991)  (sim=0.702)


## 8. Função de recomendação

**Lógica:**
1. Busca os filmes que o usuário avaliou no treino (histórico)
2. Para cada filme do histórico, encontra os TOP_K_SIMILAR mais similares
3. Pondera a similaridade pela nota dada pelo usuário (filmes bem avaliados influenciam mais)
4. Remove filmes já assistidos e retorna os TOP_N com maior score

In [8]:
def recommend(user_id: int, top_n: int = TOP_N) -> pd.DataFrame:
    if user_id not in user_idx:
        return pd.DataFrame(columns=["title", "genres", "score"])

    u = user_idx[user_id]
    user_row = train_matrix[u]

    # Filmes já avaliados (índices e notas)
    rated_indices = user_row.indices
    rated_ratings = user_row.data

    if len(rated_indices) == 0:
        return pd.DataFrame(columns=["title", "genres", "score"])

    # Acumula scores: similaridade × nota do usuário
    scores = np.zeros(item_sim.shape[0])
    for item_i, rating in zip(rated_indices, rated_ratings):
        sim_row  = item_sim[item_i]
        top_k    = np.argsort(sim_row)[::-1][1 : TOP_K_SIMILAR + 1]
        scores[top_k] += sim_row[top_k] * rating

    # Remove filmes já vistos
    scores[rated_indices] = 0

    # Top N índices
    top_indices  = np.argsort(scores)[::-1][:top_n]
    top_movie_ids = [idx_movie[i] for i in top_indices]
    top_scores    = scores[top_indices]

    result = movies[movies["movieId"].isin(top_movie_ids)].copy()
    score_map = dict(zip(top_movie_ids, top_scores))
    result["score"] = result["movieId"].map(score_map)
    return result[["title", "genres", "score"]].sort_values("score", ascending=False).reset_index(drop=True)

## 9. Avaliação do modelo

In [9]:
test_filtered = test[
    test["userId"].isin(user_idx) & test["movieId"].isin(movie_idx)
]

u_idx = test_filtered["userId"].map(user_idx).to_numpy(dtype=int)
m_idx = test_filtered["movieId"].map(movie_idx).to_numpy(dtype=int)

y_true = test_filtered["rating"].to_numpy(dtype=float)

# Predição: reconstrução da matriz centralizada + média do usuário
y_pred = np.array([
    user_factors[u] @ item_factors[m] + user_means[u]
    for u, m in zip(u_idx, m_idx)
])
y_pred = np.clip(y_pred, 0.5, 5.0)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = float(np.mean(np.abs(y_true - y_pred)))

print(f"RMSE : {rmse:.4f}  (antes da centralização: 3.4339)")
print(f"MAE  : {mae:.4f}  (antes da centralização: 3.2834)")
print(f"\n(Baseline ingênuo — média global — teria RMSE ≈ {float(y_true.std()):.4f})")

RMSE : 0.9635  (antes da centralização: 3.4339)
MAE  : 0.7177  (antes da centralização: 3.2834)

(Baseline ingênuo — média global — teria RMSE ≈ 1.0221)


In [10]:
# Precision@10: dos filmes que o usuário avaliou no teste, o modelo rankeou
# os bons (nota >= 4.0) entre os primeiros?
RELEVANCE_THRESHOLD = 4.0

# Todos os filmes avaliados no teste (relevantes e não relevantes)
test_all = (
    test_filtered
    .groupby("userId")["movieId"]
    .apply(set)
)

# Subconjunto relevante (nota >= threshold)
test_relevant = (
    test_filtered[test_filtered["rating"] >= RELEVANCE_THRESHOLD]
    .groupby("userId")["movieId"]
    .apply(set)
)

precisions = []
sample_users = test_relevant.index[:200]

for uid in tqdm(sample_users, desc="Precision@10"):
    if uid not in user_idx or uid not in test_all:
        continue

    # Candidatos = filmes que o usuário avaliou no teste e que o modelo conhece
    candidates = list(test_all[uid] & set(movie_idx.keys()))
    if not candidates:
        continue

    u     = user_idx[uid]
    c_idx = [movie_idx[mid] for mid in candidates]

    # Score do modelo (similaridade acumulada) para cada candidato
    user_row      = train_matrix[u]
    rated_indices = user_row.indices
    rated_ratings = user_row.data

    scores = np.zeros(len(candidates))
    for item_i, rating in zip(rated_indices, rated_ratings):
        sim_row = item_sim[item_i][c_idx]
        scores += sim_row * rating

    k        = min(TOP_N, len(candidates))
    top_pos  = np.argsort(scores)[::-1][:k]
    top_mids = {candidates[j] for j in top_pos}

    hits = len(top_mids & test_relevant[uid])
    precisions.append(hits / k)

print(f"Precision@{TOP_N} : {np.mean(precisions):.4f}  (candidatos = filmes avaliados no teste)")

Precision@10: 100%|██████████| 200/200 [00:00<00:00, 531.81it/s]

Precision@10 : 0.5743  (candidatos = filmes avaliados no teste)


## 10. Exemplo de recomendação

In [11]:
USER_ID = 1

# Histórico do usuário
historico = (
    train[train["userId"] == USER_ID]
    .merge(movies, on="movieId")
    .sort_values("rating", ascending=False)[["title", "genres", "rating"]]
    .head(5)
)
print(f"Top 5 filmes avaliados pelo usuário {USER_ID}:")
display(historico)

print(f"\nTop {TOP_N} recomendações para o usuário {USER_ID}:")
display(recommend(USER_ID, top_n=TOP_N))

Top 5 filmes avaliados pelo usuário 1:


,title,genres,rating
12,Lost in Translation (2003),Comedy|Drama|Romance,5.0
13,Requiem for a Dream (2000),Drama,5.0
18,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama,5.0
19,"Seventh Seal, The (Sjunde inseglet, Det) (1957)",Drama,5.0
60,Look at Me (Comme une image) (2004),Comedy|Drama|Romance,5.0



Top 10 recomendações para o usuário 1:


,title,genres,score
0,"Discreet Charm of the Bourgeoisie, The (Charme...",Comedy|Drama|Fantasy,44.742663
1,Vivre sa vie: Film en douze tableaux (My Life ...,Drama,40.607398
2,Andrei Rublev (Andrey Rublyov) (1969),Drama|War,40.274332
3,Viridiana (1961),Comedy|Drama,33.378234
4,Last Year at Marienbad (L'Année dernière à Mar...,Drama|Mystery|Romance,29.995599
5,Hiroshima Mon Amour (1959),Drama|Romance|War,27.365375
6,"Enigma of Kaspar Hauser, The (a.k.a. Mystery o...",Crime|Drama,27.294851
7,That Obscure Object of Desire (Cet obscur obje...,Drama,26.354726
8,Ali: Fear Eats the Soul (Angst essen Seele auf...,Drama|Romance,24.802047
9,"Avventura, L' (Adventure, The) (1960)",Drama|Mystery|Romance,23.905929
